# Baselines

Single-cluster predictions for Subtask 1 and all-positive predictions for Subtask 2, with the original data inspection and evaluator example.

## Setup

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "data/raw/development-swedish/usages.jsonl").is_file(), (
    "Open this notebook from the repository root."
)

for directory in (
    "data/processed/gold/SV",
    "outputs/predictions/SV",
    "outputs/evaluations",
    "outputs/exports",
):
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)


## creating dataframe

In [2]:
import pandas as pd

folder = "data/raw/development-swedish/"
df = pd.read_json(folder + "usages.jsonl", lines=True)
df.head()

,word,sentence_id,text,pos,period_label,start,end,year
0,kvinna,137d9286_62_69,nesliga lif medelst sjelfmord ; och ett stort ...,NN,1880-1883,62,69,1883
1,kvinna,b382a5e9_92_98,"— Jag sov ; jag väcktes av en kylande känsla ,...",NN,1912-1915,92,98,1914
2,kvinna,28ffffe7_42_50,"orden , så mycken fasthet öfver den gamla kvin...",NN,1880-1883,42,50,1883
3,kvinna,3754909a_117_124,Dessa klubbar ha anordnat sällskapliga samkväm...,NN,1912-1915,117,124,1913
4,kvinna,59ebb00_32_39,Den avvisningshotade eritranska kvinnan Abiba ...,NN,2006-2007,32,39,2007


#### showing shape and counts

In [3]:
print(df.shape)
df["word"].value_counts()

(1983, 8)


word
motiv        291
kvinna       290
herre        283
fru          274
fröken       174
förort       150
exportera    149
dumpa        145
suga         129
skär          98
Name: count, dtype: int64

#### grabbing word from sentence

In [4]:
first = df.iloc[0]
text = first["text"]
s = first["start"]
e = first["end"]

print(text) #full sentence
print(text[s:e]) #just word

nesliga lif medelst sjelfmord ; och ett stort antal olyckliga kvinnor , sorn sökt fly undan eländet , ha fallit i „mordeHglarnes4 händer , och deras ben vittra öfver allt i Utahs solbrända dalar .
kvinnor


#### quickly seeing how often word falls into time period

In [5]:
pd.crosstab(df["word"], df["period_label"])

period_label,1880-1883,1888-1891,1896-1899,1904-1907,1912-1915,2006-2007,2010-2011,2014-2015,2018-2019,2022-2023
word,,,,,,,,,,
dumpa,0,0,0,0,0,30,28,27,30,30
exportera,0,0,0,0,0,30,30,29,30,30
fru,29,26,23,18,28,30,30,30,30,30
fröken,28,23,30,23,23,4,9,23,7,4
förort,0,0,0,0,0,30,30,30,30,30
herre,28,24,27,27,28,29,30,30,30,30
kvinna,30,25,29,27,29,30,30,30,30,30
motiv,29,27,29,30,30,29,29,29,29,30
skär,19,14,24,16,25,0,0,0,0,0


## baselines, grabbing grading scale

In [6]:
import shutil
import sys

EVALUATOR_DIR = PROJECT_ROOT / "src/evaluation/semeval2027-task3"
BCUBED_DIR = PROJECT_ROOT / "src/evaluation/python-bcubed"
if not (EVALUATOR_DIR / "evaluation.py").is_file() or not (BCUBED_DIR / "bcubed").is_dir():
    raise FileNotFoundError("Set up the competition evaluator using the commands in README.md.")
sys.path.insert(0, str(BCUBED_DIR))
sys.path.insert(0, str(EVALUATOR_DIR))

for source in (PROJECT_ROOT / "data/raw/development-swedish").glob("subtask*.jsonl"):
    shutil.copy2(source, PROJECT_ROOT / "data/processed/gold/SV" / source.name)

#### compressing

In [7]:
from zipfile import ZipFile

with ZipFile("../outputs/exports/example.zip", "w") as archive:
    for source in Path("../data/example-submission-format").glob("*.jsonl"):
        archive.write(source, arcname=source.name)

#### grading

In [8]:
import sys
from evaluation import main as evaluate_submission

# Run the official CLI entry point in this kernel.
previous_argv = sys.argv[:]
try:
    sys.argv = [
        "evaluation.py", "outputs/exports/example.zip", "data/processed/gold",
        "-o", "outputs/evaluations/example_results.json",
    ]
    evaluate_submission()
finally:
    sys.argv = previous_argv

{
  "results": {
    "subtask1a": {
      "SV": {
        "precision": 0.6742509015772573,
        "recall": 0.11548613436982033,
        "fcubed": 0.18905192333333407
      },
      "average": {
        "precision": 0.6742509015772573,
        "recall": 0.11548613436982033,
        "fcubed": 0.18905192333333407
      }
    },
    "subtask1b": {
      "SV": 0.15291845625518283,
      "average": 0.15291845625518283
    },
    "subtask2": {
      "SV": 0.4422180107796946,
      "average": 0.4422180107796946
    }
  },
  "errors": [],
  "logs": [
    {
      "level": "info",
      "message": "Found 2 files in submission."
    },
    {
      "level": "info",
      "language": "SV",
      "subtask": "subtask1",
      "message": "Evaluation completed successfully."
    },
    {
      "level": "info",
      "language": "SV",
      "subtask": "subtask2",
      "message": "Evaluation completed successfully."
    }
  ]
}


### subtask 1 guess

In [9]:
guess1 = df[["word", "period_label", "sentence_id"]].copy()
guess1["label"] = [[0]] * len(guess1)
guess1.to_json("outputs/predictions/SV/SV_subtask1.jsonl", orient="records", lines=True)
guess1.head()

,word,period_label,sentence_id,label
0,kvinna,1880-1883,137d9286_62_69,[0]
1,kvinna,1912-1915,b382a5e9_92_98,[0]
2,kvinna,1880-1883,28ffffe7_42_50,[0]
3,kvinna,1912-1915,3754909a_117_124,[0]
4,kvinna,2006-2007,59ebb00_32_39,[0]


### subtask 2 guess (everything guessed as yes... gets 1)

In [10]:
guess2 = df[["word", "sentence_id"]].copy()
guess2["label"] = 1
guess2.to_json("outputs/predictions/SV/SV_subtask2.jsonl", orient="records", lines=True)
guess2.head()

,word,sentence_id,label
0,kvinna,137d9286_62_69,1
1,kvinna,b382a5e9_92_98,1
2,kvinna,28ffffe7_42_50,1
3,kvinna,3754909a_117_124,1
4,kvinna,59ebb00_32_39,1


### grading to see how good the guesses are

In [11]:
import sys
from pathlib import Path
sys.path.append("evaluation/semeval2027-task3")
from evaluation import evaluate_subtask1, evaluate_subtask2

score_1a, score_1b = evaluate_subtask1(Path("../data/processed/gold/SV/subtask1.jsonl"), Path(
    "../outputs/predictions/SV/SV_subtask1.jsonl"))
score_2 = evaluate_subtask2(Path("../data/processed/gold/SV/subtask2.jsonl"), Path(
    "../outputs/predictions/SV/SV_subtask2.jsonl"))

print("1a:", score_1a["fcubed"])
print("1b:", score_1b)
print("2:", score_2)

1a: 0.7563415432223628
1b: 0
2: 0.23581479994186685


/Users/kmonroygill/Desktop/semantic-change-over-time/src/evaluation/semeval2027-task3/evaluation.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pvalue = spearmanr(gold_scores, predicted_scores)
